<a href="https://colab.research.google.com/github/alexandergribenchenko/Data_Science_Finanzas/blob/main/NB_01_Creditos_Graficos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A. Librerías

In [1]:
import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
# pd.set_option('display.float_format', '{:,.2f}'.format)

# B. Funciones propias

In [2]:
def tabla_mensual_amortizacion_francesa(
    capital_inicial,
    tasa,
    periodos_n,
    periodos_temporalidad='mensual',
    tasa_temporalidad='mensual',
    tasa_tipo='nominal',
    m=12
):
    # convertir número de periodos a meses si está en años
    if periodos_temporalidad == 'mensual':
        n = periodos_n
    elif periodos_temporalidad == 'anual':
        n = periodos_n * m
    else:
        raise ValueError("periodos_temporalidad debe ser 'mensual' o 'anual'")

    # convertir tasa a tasa mensual nominal
    if tasa_temporalidad == 'mensual':
        tasa_periodo = tasa
    elif tasa_temporalidad == 'anual':
        if tasa_tipo == 'efectiva':
            tasa_periodo = (1 + tasa) ** (1 / m) - 1
        elif tasa_tipo == 'nominal':
            tasa_periodo = tasa / m
        else:
            raise ValueError("tasa_tipo debe ser 'nominal' o 'efectiva'")
    else:
        raise ValueError("tasa_temporalidad debe ser 'mensual' o 'anual'")

    # cuota fija sistema francés
    cuota = capital_inicial * (tasa_periodo * (1 + tasa_periodo) ** n) / ((1 + tasa_periodo) ** n - 1)

    df = pd.DataFrame({'periodo': range(0, n + 1)})

    saldo_insoluto = lambda k: capital_inicial * ((1 + tasa_periodo) ** n - (1 + tasa_periodo) ** k) / ((1 + tasa_periodo) ** n - 1)
    saldo_inicial = lambda k: capital_inicial if k == 1 else saldo_insoluto(k - 1)
    interes_k = lambda k: saldo_inicial(k) * tasa_periodo if k >= 1 else 0
    amortizacion_k = lambda k: cuota - interes_k(k) if k >= 1 else 0

    df['amortizacion'] = df['periodo'].apply(lambda k: round(amortizacion_k(k), 2))
    df['interes'] = df['periodo'].apply(lambda k: round(interes_k(k), 2))
    df['pago_mensual'] = df['periodo'].apply(lambda k: 0 if k == 0 else round(cuota, 2))
    df['saldo'] = df['periodo'].apply(lambda k: round(saldo_insoluto(k), 2))

    print('x = 2', float(df.amortizacion.sum()))

    return df

# 01. Prestamo original

In [25]:
df = tabla_mensual_amortizacion_francesa(
    capital_inicial=90_000_000,
    tasa=0.08688,
    periodos_n = 84,
    periodos_temporalidad = 'mensual',
    tasa_temporalidad='anual',
    tasa_tipo='efectiva'
)

fecha_inicio = '2021-01-01'

# Crear rango de fechas mensuales
fechas = pd.date_range(start=fecha_inicio, periods=len(df), freq='MS')  # 'MS' = Month Start

# Crear nueva columna con formato 'YYYY-MM'
df['periodo'] = fechas.strftime('%Y-%m')
df

x = 2 90000000.0


,periodo,amortizacion,interes,pago_mensual,saldo
0,2021-01,0.00,0.00,0.00,90000000.00
1,2021-02,791952.54,627008.06,1418960.61,89208047.46
2,2021-03,797469.88,621490.72,1418960.61,88410577.57
3,2021-04,803025.66,615934.94,1418960.61,87607551.91
4,2021-05,808620.15,610340.46,1418960.61,86798931.76
5,2021-06,814253.61,604707.00,1418960.61,85984678.16
6,2021-07,819926.31,599034.29,1418960.61,85164751.85
7,2021-08,825638.54,593322.07,1418960.61,84339113.31
8,2021-09,831390.56,587570.05,1418960.61,83507722.75
9,2021-10,837182.66,581777.95,1418960.61,82670540.09


In [33]:
df['interes_acomulado'] = df['interes'].cumsum()
df['interes_total'] = df['interes'].sum()

df['amortizacion_acomulada'] = df['amortizacion'].cumsum()
df['amortizacion_total'] = df['amortizacion'].sum()

df['pago_mensual_acomulado'] = df['pago_mensual'].cumsum()
df['pago_total_prestamo'] = df['pago_mensual'].sum()

# df['pct_interes_pago_mensual'] =  round(df['interes']/df['pago_mensual'],2)
# df['pct_amortizacion_pago_mensual'] =  round(df['amortizacion']/df['pago_mensual'],2)

# df['pct_interes_interes_total'] =  round(df['interes']/df['interes_total'],2)
# df['pct_interes_acomulado_interes_total'] =  round(df['interes_acomulado']/df['interes_total'],4)

df['pct_interes_pago_mensual'] = df['interes'] / df['pago_mensual']
df['pct_amortizacion_pago_mensual'] = df['amortizacion'] / df['pago_mensual']
df['pct_interes_interes_total'] = df['interes'] / df['interes_total']
df['pct_interes_acomulado_interes_total'] = df['interes_acomulado'] / df['interes_total']

# Normalizar acumulados a porcentaje de su total final (para que terminen en 100)
df["pct_amortizacion"] = df["amortizacion_acomulada"] / df["amortizacion_total"] * 100
df["pct_interes"] = df["interes_acomulado"] / df["interes_total"] * 100
df["pct_pago"] = df["pago_mensual_acomulado"] / df["pago_total_prestamo"] * 100

columns_interest = ['periodo',
                    'pago_mensual', 'pago_mensual_acomulado', 'pago_total_prestamo',
                    'amortizacion', 'amortizacion_acomulada','amortizacion_total',
                    'interes', 'interes_acomulado','interes_total',
                    'pct_interes_pago_mensual', 'pct_amortizacion_pago_mensual',
                    'pct_interes_interes_total', 'pct_interes_acomulado_interes_total']
df = df[columns_interest].reset_index(drop=False).rename(columns={'index':'cuota'})
df['periodo'] = pd.to_datetime(df['periodo'])

money_cols = [
    "pago_mensual", "pago_mensual_acomulado", "pago_total_prestamo",
    "amortizacion", "amortizacion_acomulada", "amortizacion_total",
    "interes", "interes_acomulado", "interes_total"
]

rate_cols = [
    "pct_interes_pago_mensual",
    "pct_amortizacion_pago_mensual",
    "pct_interes_interes_total",
    "pct_interes_acomulado_interes_total"
]

money_fmt = "{:,.2f}"
rate_fmt = "{:,.4f}"

fmt_dict = {col: money_fmt for col in money_cols} | {col: rate_fmt for col in rate_cols}

df.style.format(fmt_dict)

,cuota,periodo,pago_mensual,pago_mensual_acomulado,pago_total_prestamo,amortizacion,amortizacion_acomulada,amortizacion_total,interes,interes_acomulado,interes_total,pct_interes_pago_mensual,pct_amortizacion_pago_mensual,pct_interes_interes_total,pct_interes_acomulado_interes_total
0,0,2021-01-01 00:00:00,0.00,0.00,"119,192,691.24",0.00,0.00,"90,000,000.00",0.00,0.00,"29,192,690.95",nan,nan,0.0000,0.0000
1,1,2021-02-01 00:00:00,"1,418,960.61","1,418,960.61","119,192,691.24","791,952.54","791,952.54","90,000,000.00","627,008.06","627,008.06","29,192,690.95",0.4419,0.5581,0.0215,0.0215
2,2,2021-03-01 00:00:00,"1,418,960.61","2,837,921.22","119,192,691.24","797,469.88","1,589,422.42","90,000,000.00","621,490.72","1,248,498.78","29,192,690.95",0.4380,0.5620,0.0213,0.0428
3,3,2021-04-01 00:00:00,"1,418,960.61","4,256,881.83","119,192,691.24","803,025.66","2,392,448.08","90,000,000.00","615,934.94","1,864,433.72","29,192,690.95",0.4341,0.5659,0.0211,0.0639
4,4,2021-05-01 00:00:00,"1,418,960.61","5,675,842.44","119,192,691.24","808,620.15","3,201,068.23","90,000,000.00","610,340.46","2,474,774.18","29,192,690.95",0.4301,0.5699,0.0209,0.0848
5,5,2021-06-01 00:00:00,"1,418,960.61","7,094,803.05","119,192,691.24","814,253.61","4,015,321.84","90,000,000.00","604,707.00","3,079,481.18","29,192,690.95",0.4262,0.5738,0.0207,0.1055
6,6,2021-07-01 00:00:00,"1,418,960.61","8,513,763.66","119,192,691.24","819,926.31","4,835,248.15","90,000,000.00","599,034.29","3,678,515.47","29,192,690.95",0.4222,0.5778,0.0205,0.1260
7,7,2021-08-01 00:00:00,"1,418,960.61","9,932,724.27","119,192,691.24","825,638.54","5,660,886.69","90,000,000.00","593,322.07","4,271,837.54","29,192,690.95",0.4181,0.5819,0.0203,0.1463
8,8,2021-09-01 00:00:00,"1,418,960.61","11,351,684.88","119,192,691.24","831,390.56","6,492,277.25","90,000,000.00","587,570.05","4,859,407.59","29,192,690.95",0.4141,0.5859,0.0201,0.1665
9,9,2021-10-01 00:00:00,"1,418,960.61","12,770,645.49","119,192,691.24","837,182.66","7,329,459.91","90,000,000.00","581,777.95","5,441,185.54","29,192,690.95",0.4100,0.5900,0.0199,0.1864


In [34]:
import plotly.graph_objects as go
df_plot = df.copy()

# Creamos un hovertext global con Fecha + Cuota
fig = go.Figure()

# Definimos la cabecera única con fecha y cuota
custom_header = (
    "Periodo: %{x|%Y-%m}<br>" +  # Fecha en formato Año-Mes
    "Cuota: %{customdata}<br>"
)

# % Capital
fig.add_trace(go.Scatter(
    x=df_plot["periodo"],
    y=df_plot["pct_amortizacion"],
    mode="lines+markers",
    name="% Capital acumulado",
    customdata=df_plot["cuota"],  # pasamos la cuota
    hovertemplate=custom_header + "%{y:.2f}%<extra></extra>",
    line=dict(color="seagreen")
))

# % Intereses
fig.add_trace(go.Scatter(
    x=df_plot["periodo"],
    y=df_plot["pct_interes"],
    mode="lines+markers",
    name="% Interés acumulado",
    customdata=df_plot["cuota"],
    hovertemplate=custom_header + "%{y:.2f}%<extra></extra>",
    line=dict(color="tomato")
))

# % Pago total
fig.add_trace(go.Scatter(
    x=df_plot["periodo"],
    y=df_plot["pct_pago"],
    mode="lines+markers",
    name="% Pago total acumulado",
    customdata=df_plot["cuota"],
    hovertemplate=custom_header + "%{y:.2f}%<extra></extra>",
    line=dict(color="royalblue")
))

# Layout
fig.update_layout(
    title="% de avance relativo de cada componente (cada uno normalizado a 100%)",
    xaxis=dict(title="Periodo", tickformat="%Y-%m"),
    yaxis=dict(title="Porcentaje acumulado (%)", range=[0, 105], showgrid=True),
    hovermode="x unified",
    hoverlabel=dict(bgcolor="white", font_size=12),
    legend=dict(x=0.01, y=0.99, bgcolor="rgba(255,255,255,0.8)")
)

fig.show()


KeyError: 'pct_amortizacion'

In [17]:
import plotly.graph_objects as go

# Creamos un hovertext global con Fecha + Cuota
df_plot["hover_info"] = (
    "Periodo: " + df_plot["periodo"].astype(str) +
    "<br>Cuota: " + df_plot["cuota"].astype(str)
)

fig = go.Figure()

# % Capital
fig.add_trace(go.Scatter(
    x=df_plot["periodo"],
    y=df_plot["pct_amortizacion"],
    mode="lines+markers",
    name="% Capital acumulado",
    hovertemplate="%{y:.2f}%<extra></extra>",
    line=dict(color="seagreen")
))

# % Intereses
fig.add_trace(go.Scatter(
    x=df_plot["periodo"],
    y=df_plot["pct_interes"],
    mode="lines+markers",
    name="% Interés acumulado",
    hovertemplate="%{y:.2f}%<extra></extra>",
    line=dict(color="tomato")
))

# % Pago total
fig.add_trace(go.Scatter(
    x=df_plot["periodo"],
    y=df_plot["pct_pago"],
    mode="lines+markers",
    name="% Pago total acumulado",
    hovertemplate="%{y:.2f}%<extra></extra>",
    line=dict(color="royalblue")
))

# Layout con hover unificado
fig.update_layout(
    title="% de avance relativo de cada componente (cada uno normalizado a 100%)",
    xaxis=dict(title="Periodo", tickformat="%Y-%m"),
    yaxis=dict(title="Porcentaje acumulado (%)", range=[0, 105], showgrid=True),
    hovermode="x unified",
    hoverlabel=dict(bgcolor="white", font_size=12),
    legend=dict(x=0.01, y=0.99, bgcolor="rgba(255,255,255,0.8)")
)

# Truco: mostrar solo una vez la info general (fecha + cuota)
fig.update_layout(
    hovermode="x unified",
    hoverlabel=dict(namelength=0)
)
fig.update_traces(hoverinfo="text+name")

# Inyectamos el texto global en cada punto, pero aparecerá solo una vez en el encabezado
for trace in fig.data:
    trace.text = df_plot["hover_info"]

fig.show()


In [16]:
fig = go.Figure()

# % Capital
fig.add_trace(go.Scatter(
    x=df_plot["periodo"],
    y=df_plot["pct_amortizacion"],
    mode="lines+markers",
    name="% Capital acumulado",
    customdata=df_plot["cuota"],
    hovertemplate="%{y:.2f}%<extra></extra>",  # solo valor
    line=dict(color="seagreen")
))

# % Intereses
fig.add_trace(go.Scatter(
    x=df_plot["periodo"],
    y=df_plot["pct_interes"],
    mode="lines+markers",
    name="% Interés acumulado",
    customdata=df_plot["cuota"],
    hovertemplate="%{y:.2f}%<extra></extra>",  # solo valor
    line=dict(color="tomato")
))

# % Pago total
fig.add_trace(go.Scatter(
    x=df_plot["periodo"],
    y=df_plot["pct_pago"],
    mode="lines+markers",
    name="% Pago total acumulado",
    customdata=df_plot["cuota"],
    hovertemplate="%{y:.2f}%<extra></extra>",  # solo valor
    line=dict(color="royalblue")
))

# Layout con hover unificado
fig.update_layout(
    title="% de avance relativo de cada componente (cada uno normalizado a 100%)",
    xaxis=dict(title="Periodo"),
    yaxis=dict(
        title="Porcentaje acumulado (%)",
        range=[0, 105],
        showgrid=True,
        zeroline=False
    ),
    hovermode="x unified",
    legend=dict(x=0.01, y=0.99, bgcolor="rgba(255,255,255,0.8)")
)

# Personalizar cabecera del hover unificado (fecha + cuota)
fig.update_traces(customdata=df_plot["cuota"])
fig.update_layout(
    hoverlabel=dict(bgcolor="white", font_size=12),
    xaxis=dict(
        tickformat="%Y-%m",
        title="Periodo"
    )
)

fig.show()


In [15]:
import plotly.graph_objects as go

# Filtrar desde la cuota 1 en adelante
df_plot = df.iloc[1:].copy()

# Totales finales
total_amortizacion = df["amortizacion_acomulada"].iloc[-1]
total_interes = df["interes_acomulado"].iloc[-1]
total_pago = df["pago_mensual_acomulado"].iloc[-1]

# Normalización
df_plot["pct_amortizacion"] = df_plot["amortizacion_acomulada"] / total_amortizacion * 100
df_plot["pct_interes"] = df_plot["interes_acomulado"] / total_interes * 100
df_plot["pct_pago"] = df_plot["pago_mensual_acomulado"] / total_pago * 100

fig = go.Figure()

# % Capital
fig.add_trace(go.Scatter(
    x=df_plot["periodo"],
    y=df_plot["pct_amortizacion"],
    mode="lines+markers",
    name="% Capital acumulado",
    customdata=df_plot["cuota"],
    hovertemplate="Periodo: %{x|%Y-%m}<br>" +
                  "Cuota: %{customdata}<br>" +
                  "% Capital acumulado: %{y:.2f}%<extra></extra>",
    line=dict(color="seagreen")
))

# % Intereses
fig.add_trace(go.Scatter(
    x=df_plot["periodo"],
    y=df_plot["pct_interes"],
    mode="lines+markers",
    name="% Interés acumulado",
    customdata=df_plot["cuota"],
    hovertemplate="Periodo: %{x|%Y-%m}<br>" +
                  "Cuota: %{customdata}<br>" +
                  "% Interés acumulado: %{y:.2f}%<extra></extra>",
    line=dict(color="tomato")
))

# % Pago total
fig.add_trace(go.Scatter(
    x=df_plot["periodo"],
    y=df_plot["pct_pago"],
    mode="lines+markers",
    name="% Pago total acumulado",
    customdata=df_plot["cuota"],
    hovertemplate="Periodo: %{x|%Y-%m}<br>" +
                  "Cuota: %{customdata}<br>" +
                  "% Pago acumulado: %{y:.2f}%<extra></extra>",
    line=dict(color="royalblue")
))

# Layout
fig.update_layout(
    title="% de avance relativo de cada componente (cada uno normalizado a 100%)",
    xaxis=dict(title="Periodo"),
    yaxis=dict(
        title="Porcentaje acumulado (%)",
        range=[0, 105],
        showgrid=True,
        zeroline=False
    ),
    hovermode="x unified",
    legend=dict(x=0.01, y=0.99, bgcolor="rgba(255,255,255,0.8)")
)

fig.show()


In [14]:
import plotly.graph_objects as go

# Filtrar desde la cuota 1 en adelante (omitimos la inicial con 0s)
df_plot = df.iloc[1:].copy()

# Normalización contra el total pagado al final
total_pagado = df["pago_total_prestamo"].iloc[0]

df_plot["pct_amortizacion_total_pago"] = df_plot["amortizacion_acomulada"] / total_pagado * 100
df_plot["pct_interes_total_pago"] = df_plot["interes_acomulado"] / total_pagado * 100
df_plot["pct_pago_total_pago"] = df_plot["pago_mensual_acomulado"] / total_pagado * 100

fig = go.Figure()

# % Capital acumulado sobre pago total
fig.add_trace(go.Scatter(
    x=df_plot["periodo"],
    y=df_plot["pct_amortizacion_total_pago"],
    mode="lines+markers",
    name="% Capital acumulado",
    customdata=df_plot["cuota"],
    hovertemplate="Periodo: %{x|%Y-%m}<br>" +
                  "Cuota: %{customdata}<br>" +
                  "% Capital (vs total pagado): %{y:.2f}%<extra></extra>",
    line=dict(color="seagreen")
))

# % Intereses acumulados sobre pago total
fig.add_trace(go.Scatter(
    x=df_plot["periodo"],
    y=df_plot["pct_interes_total_pago"],
    mode="lines+markers",
    name="% Interés acumulado",
    customdata=df_plot["cuota"],
    hovertemplate="Periodo: %{x|%Y-%m}<br>" +
                  "Cuota: %{customdata}<br>" +
                  "% Interés (vs total pagado): %{y:.2f}%<extra></extra>",
    line=dict(color="tomato")
))

# % Total pagado (capital + interés)
fig.add_trace(go.Scatter(
    x=df_plot["periodo"],
    y=df_plot["pct_pago_total_pago"],
    mode="lines+markers",
    name="% Total pagado",
    customdata=df_plot["cuota"],
    hovertemplate="Periodo: %{x|%Y-%m}<br>" +
                  "Cuota: %{customdata}<br>" +
                  "% Total acumulado: %{y:.2f}%<extra></extra>",
    line=dict(color="royalblue")
))

# Layout
fig.update_layout(
    title="% de avance del crédito frente al total pagado (capital + intereses)",
    xaxis=dict(title="Periodo"),
    yaxis=dict(
        title="Porcentaje del total pagado (%)",
        range=[0, 105],  # para marcar el 100%
        showgrid=True,
        zeroline=False
    ),
    hovermode="x unified",
    legend=dict(x=0.01, y=0.99, bgcolor="rgba(255,255,255,0.8)")
)

fig.show()


In [13]:
import plotly.graph_objects as go

# Asegurar que periodo sea datetime
df['periodo'] = pd.to_datetime(df['periodo'])

# Omitir la primera fila (cuota 0 con NaN en %)
df_plot = df.iloc[1:]

fig = go.Figure()

# Amortización (eje izquierdo)
fig.add_trace(go.Scatter(
    x=df_plot['periodo'],
    y=df_plot['amortizacion'],
    mode="lines+markers",
    name="Amortización",
    customdata=df_plot['cuota'],
    hovertemplate="Periodo: %{x|%Y-%m}<br>" +
                  "Cuota: %{customdata}<br>" +
                  "Amortización: %{y:,.2f}<extra></extra>",
    line=dict(color="seagreen")
))

# Interés (eje izquierdo)
fig.add_trace(go.Scatter(
    x=df_plot['periodo'],
    y=df_plot['interes'],
    mode="lines+markers",
    name="Interés",
    customdata=df_plot['cuota'],
    hovertemplate="Periodo: %{x|%Y-%m}<br>" +
                  "Cuota: %{customdata}<br>" +
                  "Interés: %{y:,.2f}<extra></extra>",
    line=dict(color="tomato")
))

# % Amortización sobre pago mensual (eje derecho)
fig.add_trace(go.Scatter(
    x=df_plot['periodo'],
    y=df_plot['pct_amortizacion_pago_mensual']*100,
    mode="lines",
    name="% Amortización / Pago mensual",
    customdata=df_plot['cuota'],
    hovertemplate="Periodo: %{x|%Y-%m}<br>" +
                  "Cuota: %{customdata}<br>" +
                  "% Amortización: %{y:.2f}%<extra></extra>",
    line=dict(color="seagreen", dash="dot"),
    yaxis="y2"
))

# % Interés sobre pago mensual (eje derecho)
fig.add_trace(go.Scatter(
    x=df_plot['periodo'],
    y=df_plot['pct_interes_pago_mensual']*100,
    mode="lines",
    name="% Interés / Pago mensual",
    customdata=df_plot['cuota'],
    hovertemplate="Periodo: %{x|%Y-%m}<br>" +
                  "Cuota: %{customdata}<br>" +
                  "% Interés: %{y:.2f}%<extra></extra>",
    line=dict(color="tomato", dash="dot"),
    yaxis="y2"
))

# Layout
fig.update_layout(
    title="Amortización e Interés (valores y %)",
    xaxis=dict(title="Periodo"),
    yaxis=dict(
        title="Valores absolutos",
        showgrid=True,
        zeroline=False
    ),
    yaxis2=dict(
        title="Porcentaje",
        overlaying="y",
        side="right",
        showgrid=False
    ),
    hovermode="x unified",
    legend=dict(x=0.01, y=0.99, bgcolor="rgba(255,255,255,0.8)")
)

fig.show()


In [12]:
import plotly.graph_objects as go

df['periodo'] = pd.to_datetime(df['periodo'])

fig = go.Figure()

# Amortización (eje izquierdo)
fig.add_trace(go.Scatter(
    x=df['periodo'],
    y=df['amortizacion'],
    mode="lines+markers",
    name="Amortización",
    customdata=df['cuota'],
    hovertemplate="Periodo: %{x|%Y-%m}<br>" +
                  "Cuota: %{customdata}<br>" +
                  "Amortización: %{y:,.2f}<extra></extra>",
    line=dict(color="seagreen")
))

# Interés (eje izquierdo)
fig.add_trace(go.Scatter(
    x=df['periodo'],
    y=df['interes'],
    mode="lines+markers",
    name="Interés",
    customdata=df['cuota'],
    hovertemplate="Periodo: %{x|%Y-%m}<br>" +
                  "Cuota: %{customdata}<br>" +
                  "Interés: %{y:,.2f}<extra></extra>",
    line=dict(color="tomato")
))

# % Amortización sobre pago mensual (eje derecho)
fig.add_trace(go.Scatter(
    x=df['periodo'],
    y=df['pct_amortizacion_pago_mensual']*100,
    mode="lines",
    name="% Amortización / Pago mensual",
    customdata=df['cuota'],
    hovertemplate="Periodo: %{x|%Y-%m}<br>" +
                  "Cuota: %{customdata}<br>" +
                  "% Amortización: %{y:.2f}%<extra></extra>",
    line=dict(color="seagreen", dash="dot"),
    yaxis="y2"
))

# % Interés sobre pago mensual (eje derecho)
fig.add_trace(go.Scatter(
    x=df['periodo'],
    y=df['pct_interes_pago_mensual']*100,
    mode="lines",
    name="% Interés / Pago mensual",
    customdata=df['cuota'],
    hovertemplate="Periodo: %{x|%Y-%m}<br>" +
                  "Cuota: %{customdata}<br>" +
                  "% Interés: %{y:.2f}%<extra></extra>",
    line=dict(color="tomato", dash="dot"),
    yaxis="y2"
))

# Layout
fig.update_layout(
    title="Amortización e Interés (valores y %)",
    xaxis=dict(title="Periodo"),
    yaxis=dict(
        title="Valores absolutos",
        showgrid=True,
        zeroline=False
    ),
    yaxis2=dict(
        title="Porcentaje",
        overlaying="y",
        side="right",
        showgrid=False
    ),
    hovermode="x unified",
    legend=dict(x=0.01, y=0.99, bgcolor="rgba(255,255,255,0.8)")
)

fig.show()


In [11]:
import plotly.graph_objects as go

# Asegurar que periodo es datetime para formatear bonito
df['periodo'] = pd.to_datetime(df['periodo'])

fig = go.Figure()

# Ejemplo con Amortización
fig.add_trace(go.Scatter(
    x=df['periodo'],
    y=df['amortizacion'],
    mode="lines+markers",
    name="Amortización",
    customdata=df['cuota'],  # Pasamos la cuota como dato extra
    hovertemplate="Periodo: %{x|%Y-%m}<br>" +  # fecha en hover
                  "Cuota: %{customdata}<br>" +  # número de cuota
                  "Amortización: %{y:,.2f}<extra></extra>"
))

# Otro ejemplo con Interés
fig.add_trace(go.Scatter(
    x=df['periodo'],
    y=df['interes'],
    mode="lines+markers",
    name="Interés",
    customdata=df['cuota'],
    hovertemplate="Periodo: %{x|%Y-%m}<br>" +
                  "Cuota: %{customdata}<br>" +
                  "Interés: %{y:,.2f}<extra></extra>"
))

fig.update_layout(
    title="Amortización e Interés",
    xaxis_title="Periodo",
    yaxis_title="Valor",
    hovermode="x unified"
)

fig.show()


In [10]:
import plotly.graph_objects as go

# ✅ Opción 1: eje X = periodo, hover = cuota
fig1 = go.Figure()
fig1.add_trace(go.Bar(
    x=df['periodo'], y=df['interes'],
    name="Interés", marker_color="tomato",
    hovertext="Cuota: " + df['cuota'].astype(str)
))
fig1.add_trace(go.Bar(
    x=df['periodo'], y=df['amortizacion'],
    name="Amortización", marker_color="seagreen",
    hovertext="Cuota: " + df['cuota'].astype(str)
))
fig1.update_layout(
    barmode="stack",
    title="Composición del Pago Mensual",
    xaxis_title="Periodo",
    yaxis_title="Monto ($)",
    template="plotly_white"
)

# ✅ Opción 2: eje X = cuota, hover = periodo
fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=df['cuota'], y=df['interes_acomulado'],
    mode="lines+markers", name="Interés acumulado",
    line=dict(color="tomato", dash="dot"),
    hovertext=df['periodo'].dt.strftime("%Y-%m")
))
fig2.add_trace(go.Scatter(
    x=df['cuota'], y=df['amortizacion_acomulada'],
    mode="lines+markers", name="Amortización acumulada",
    line=dict(color="seagreen"),
    hovertext=df['periodo'].dt.strftime("%Y-%m")
))

# Totales como asíntotas
fig2.add_hline(y=df['interes_total'].iloc[0], line=dict(color="tomato", dash="dash"),
               annotation_text="Total Interés", annotation_position="top left")
fig2.add_hline(y=df['amortizacion_total'].iloc[0], line=dict(color="seagreen", dash="dash"),
               annotation_text="Total Amortización", annotation_position="top left")
fig2.add_hline(y=df['pago_total_prestamo'].iloc[0], line=dict(color="black", dash="dash"),
               annotation_text="Total Préstamo", annotation_position="top left")

fig2.update_layout(
    title="Evolución Acumulada y Totales",
    xaxis_title="Cuota",
    yaxis_title="Monto ($)",
    template="plotly_white"
)

# ✅ Opción 3: Tasas (%)
fig3 = go.Figure()
fig3.add_trace(go.Scatter(
    x=df['cuota'], y=df['pct_interes_pago_mensual'],
    mode="lines+markers", name="% Interés/Pago mensual",
    line=dict(color="orange"),
    hovertext=df['periodo'].dt.strftime("%Y-%m")
))
fig3.add_trace(go.Scatter(
    x=df['cuota'], y=df['pct_amortizacion_pago_mensual'],
    mode="lines+markers", name="% Amortización/Pago mensual",
    line=dict(color="blue"),
    hovertext=df['periodo'].dt.strftime("%Y-%m")
))
fig3.add_trace(go.Scatter(
    x=df['cuota'], y=df['pct_interes_interes_total'],
    mode="lines+markers", name="% Interés/Total interés",
    line=dict(color="red", dash="dot"),
    hovertext=df['periodo'].dt.strftime("%Y-%m")
))
fig3.add_trace(go.Scatter(
    x=df['cuota'], y=df['pct_interes_acomulado_interes_total'],
    mode="lines+markers", name="% Interés acumulado/Total interés",
    line=dict(color="purple", dash="dot"),
    hovertext=df['periodo'].dt.strftime("%Y-%m")
))

fig3.update_layout(
    title="Evolución de Tasas y Proporciones",
    xaxis_title="Cuota",
    yaxis_title="Proporción (%)",
    yaxis_tickformat=".2%",
    template="plotly_white"
)

# Mostrar
fig1.show()
fig2.show()
fig3.show()


In [6]:
import plotly.graph_objects as go

# Asegúrate de que 'periodo' sea string o datetime para el eje X
df['periodo'] = pd.to_datetime(df['periodo'])

fig = go.Figure()

# --- Gráfico 1: composición del pago mensual ---
fig.add_trace(go.Bar(
    x=df['periodo'],
    y=df['interes'],
    name='Interés',
    marker_color='tomato'
))
fig.add_trace(go.Bar(
    x=df['periodo'],
    y=df['amortizacion'],
    name='Amortización',
    marker_color='seagreen'
))

# --- Gráfico 2: líneas acumuladas ---
fig.add_trace(go.Scatter(
    x=df['periodo'],
    y=df['interes_acomulado'],
    mode='lines+markers',
    name='Interés acumulado',
    line=dict(color='tomato', width=2, dash='dot')
))
fig.add_trace(go.Scatter(
    x=df['periodo'],
    y=df['amortizacion_acomulada'],
    mode='lines+markers',
    name='Amortización acumulada',
    line=dict(color='seagreen', width=2)
))

fig.update_layout(
    barmode='stack',
    title='Evolución del préstamo',
    xaxis_title='Periodo',
    yaxis_title='Monto',
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    hovermode="x unified",
    template="plotly_white"
)

fig.show()

In [7]:
import plotly.graph_objects as go

# Convertir periodo a fecha si no lo está
df['periodo'] = pd.to_datetime(df['periodo'])

fig = go.Figure()

# --- Barras apiladas (pago mensual) ---
fig.add_trace(go.Bar(
    x=df['periodo'],
    y=df['interes'],
    name="Interés (pago mensual)",
    marker_color="tomato",
    hovertemplate="Periodo: %{x}<br>Interés: %{y:,.2f}<extra></extra>",
    yaxis="y1"
))
fig.add_trace(go.Bar(
    x=df['periodo'],
    y=df['amortizacion'],
    name="Amortización (pago mensual)",
    marker_color="seagreen",
    hovertemplate="Periodo: %{x}<br>Amortización: %{y:,.2f}<extra></extra>",
    yaxis="y1"
))

# --- Líneas acumuladas ---
fig.add_trace(go.Scatter(
    x=df['periodo'],
    y=df['interes_acomulado'],
    mode="lines+markers",
    name="Interés acumulado",
    line=dict(color="tomato", dash="dot"),
    yaxis="y1"
))
fig.add_trace(go.Scatter(
    x=df['periodo'],
    y=df['amortizacion_acomulada'],
    mode="lines+markers",
    name="Amortización acumulada",
    line=dict(color="seagreen"),
    yaxis="y1"
))

# --- Líneas de porcentajes ---
fig.add_trace(go.Scatter(
    x=df['periodo'],
    y=df['pct_interes_pago_mensual'],
    mode="lines",
    name="% Interés / Pago mensual",
    line=dict(color="orange", width=2),
    yaxis="y2"
))
fig.add_trace(go.Scatter(
    x=df['periodo'],
    y=df['pct_amortizacion_pago_mensual'],
    mode="lines",
    name="% Amortización / Pago mensual",
    line=dict(color="blue", width=2),
    yaxis="y2"
))

# --- Layout ---
fig.update_layout(
    title="Evolución del préstamo: pagos, acumulados y porcentajes",
    barmode="stack",
    xaxis=dict(title="Periodo"),
    yaxis=dict(
        title="Montos ($)",
        side="left",
        showgrid=True
    ),
    yaxis2=dict(
        title="Porcentaje del pago mensual",
        overlaying="y",
        side="right",
        tickformat=".0%",
        showgrid=False
    ),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    hovermode="x unified",
    template="plotly_white",
    height=600
)

fig.show()


In [8]:
import plotly.graph_objects as go

# Asegurar fechas en el eje X
df['periodo'] = pd.to_datetime(df['periodo'])

fig = go.Figure()

# --- Barras apiladas: pago mensual ---
fig.add_trace(go.Bar(
    x=df['periodo'], y=df['interes'],
    name="Interés (pago mensual)", marker_color="tomato",
    yaxis="y1"
))
fig.add_trace(go.Bar(
    x=df['periodo'], y=df['amortizacion'],
    name="Amortización (pago mensual)", marker_color="seagreen",
    yaxis="y1"
))

# --- Líneas acumuladas ---
fig.add_trace(go.Scatter(
    x=df['periodo'], y=df['interes_acomulado'],
    mode="lines+markers", name="Interés acumulado",
    line=dict(color="tomato", dash="dot"), yaxis="y1"
))
fig.add_trace(go.Scatter(
    x=df['periodo'], y=df['amortizacion_acomulada'],
    mode="lines+markers", name="Amortización acumulada",
    line=dict(color="seagreen"), yaxis="y1"
))

# --- Líneas horizontales de totales ---
fig.add_hline(y=df['interes_total'].iloc[0],
              line=dict(color="tomato", dash="dash"),
              annotation_text="Total Interés", annotation_position="top left")
fig.add_hline(y=df['amortizacion_total'].iloc[0],
              line=dict(color="seagreen", dash="dash"),
              annotation_text="Total Amortización", annotation_position="top left")
fig.add_hline(y=df['pago_total_prestamo'].iloc[0],
              line=dict(color="black", dash="dash"),
              annotation_text="Total Préstamo", annotation_position="top left")

# --- Tasas (eje derecho) ---
fig.add_trace(go.Scatter(
    x=df['periodo'], y=df['pct_interes_pago_mensual'],
    mode="lines", name="% Interés/Pago", line=dict(color="orange", width=2),
    yaxis="y2"
))
fig.add_trace(go.Scatter(
    x=df['periodo'], y=df['pct_amortizacion_pago_mensual'],
    mode="lines", name="% Amortización/Pago", line=dict(color="blue", width=2),
    yaxis="y2"
))
fig.add_trace(go.Scatter(
    x=df['periodo'], y=df['pct_interes_interes_total'],
    mode="lines", name="% Interés/Total Interés", line=dict(color="red", width=2, dash="dot"),
    yaxis="y2"
))
fig.add_trace(go.Scatter(
    x=df['periodo'], y=df['pct_interes_acomulado_interes_total'],
    mode="lines", name="% Interés Acumulado/Total", line=dict(color="purple", width=2, dash="dot"),
    yaxis="y2"
))

# --- Layout ---
fig.update_layout(
    title="Evolución completa del préstamo",
    barmode="stack",
    xaxis=dict(title="Periodo"),
    yaxis=dict(
        title="Montos ($)",
        side="left",
        showgrid=True
    ),
    yaxis2=dict(
        title="Proporciones (%)",
        overlaying="y",
        side="right",
        tickformat=".2%",
        showgrid=False
    ),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    hovermode="x unified",
    template="plotly_white",
    height=700
)

fig.show()


In [9]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Asegurar fechas
df['periodo'] = pd.to_datetime(df['periodo'])

# --- 1. Composición del pago mensual ---
fig1 = go.Figure()
fig1.add_trace(go.Bar(x=df['periodo'], y=df['interes'],
                      name="Interés", marker_color="tomato"))
fig1.add_trace(go.Bar(x=df['periodo'], y=df['amortizacion'],
                      name="Amortización", marker_color="seagreen"))
fig1.update_layout(
    barmode="stack",
    title="Composición del Pago Mensual",
    xaxis_title="Periodo",
    yaxis_title="Monto ($)",
    template="plotly_white"
)

# --- 2. Acumulados con totales ---
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=df['periodo'], y=df['interes_acomulado'],
                          mode="lines+markers", name="Interés acumulado",
                          line=dict(color="tomato", dash="dot")))
fig2.add_trace(go.Scatter(x=df['periodo'], y=df['amortizacion_acomulada'],
                          mode="lines+markers", name="Amortización acumulada",
                          line=dict(color="seagreen")))

# Líneas horizontales de totales
fig2.add_hline(y=df['interes_total'].iloc[0], line=dict(color="tomato", dash="dash"),
               annotation_text="Total Interés", annotation_position="top left")
fig2.add_hline(y=df['amortizacion_total'].iloc[0], line=dict(color="seagreen", dash="dash"),
               annotation_text="Total Amortización", annotation_position="top left")
fig2.add_hline(y=df['pago_total_prestamo'].iloc[0], line=dict(color="black", dash="dash"),
               annotation_text="Total Préstamo", annotation_position="top left")

fig2.update_layout(
    title="Evolución Acumulada y Totales",
    xaxis_title="Periodo",
    yaxis_title="Monto ($)",
    template="plotly_white"
)

# --- 3. Tasas (% proporciones) ---
fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=df['periodo'], y=df['pct_interes_pago_mensual'],
                          mode="lines+markers", name="% Interés/Pago mensual",
                          line=dict(color="orange")))
fig3.add_trace(go.Scatter(x=df['periodo'], y=df['pct_amortizacion_pago_mensual'],
                          mode="lines+markers", name="% Amortización/Pago mensual",
                          line=dict(color="blue")))
fig3.add_trace(go.Scatter(x=df['periodo'], y=df['pct_interes_interes_total'],
                          mode="lines+markers", name="% Interés/Total interés",
                          line=dict(color="red", dash="dot")))
fig3.add_trace(go.Scatter(x=df['periodo'], y=df['pct_interes_acomulado_interes_total'],
                          mode="lines+markers", name="% Interés acumulado/Total interés",
                          line=dict(color="purple", dash="dot")))

fig3.update_layout(
    title="Evolución de Tasas y Proporciones",
    xaxis_title="Periodo",
    yaxis_title="Proporción (%)",
    yaxis_tickformat=".2%",
    template="plotly_white"
)

# Mostrar
fig1.show()
fig2.show()
fig3.show()


In [5]:
import plotly.graph_objects as go

# ✅ Opción 1: eje X = periodo, hover = cuota
fig1 = go.Figure()
fig1.add_trace(go.Bar(
    x=df['periodo'], y=df['interes'],
    name="Interés", marker_color="tomato",
    hovertext="Cuota: " + df['cuota'].astype(str)
))
fig1.add_trace(go.Bar(
    x=df['periodo'], y=df['amortizacion'],
    name="Amortización", marker_color="seagreen",
    hovertext="Cuota: " + df['cuota'].astype(str)
))
fig1.update_layout(
    barmode="stack",
    title="Composición del Pago Mensual",
    xaxis_title="Periodo",
    yaxis_title="Monto ($)",
    template="plotly_white"
)

# ✅ Opción 2: eje X = cuota, hover = periodo
fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=df['cuota'], y=df['interes_acomulado'],
    mode="lines+markers", name="Interés acumulado",
    line=dict(color="tomato", dash="dot"),
    hovertext=df['periodo'].dt.strftime("%Y-%m")
))
fig2.add_trace(go.Scatter(
    x=df['cuota'], y=df['amortizacion_acomulada'],
    mode="lines+markers", name="Amortización acumulada",
    line=dict(color="seagreen"),
    hovertext=df['periodo'].dt.strftime("%Y-%m")
))

# Totales como asíntotas
fig2.add_hline(y=df['interes_total'].iloc[0], line=dict(color="tomato", dash="dash"),
               annotation_text="Total Interés", annotation_position="top left")
fig2.add_hline(y=df['amortizacion_total'].iloc[0], line=dict(color="seagreen", dash="dash"),
               annotation_text="Total Amortización", annotation_position="top left")
fig2.add_hline(y=df['pago_total_prestamo'].iloc[0], line=dict(color="black", dash="dash"),
               annotation_text="Total Préstamo", annotation_position="top left")

fig2.update_layout(
    title="Evolución Acumulada y Totales",
    xaxis_title="Cuota",
    yaxis_title="Monto ($)",
    template="plotly_white"
)

# ✅ Opción 3: Tasas (%)
fig3 = go.Figure()
fig3.add_trace(go.Scatter(
    x=df['cuota'], y=df['pct_interes_pago_mensual'],
    mode="lines+markers", name="% Interés/Pago mensual",
    line=dict(color="orange"),
    hovertext=df['periodo'].dt.strftime("%Y-%m")
))
fig3.add_trace(go.Scatter(
    x=df['cuota'], y=df['pct_amortizacion_pago_mensual'],
    mode="lines+markers", name="% Amortización/Pago mensual",
    line=dict(color="blue"),
    hovertext=df['periodo'].dt.strftime("%Y-%m")
))
fig3.add_trace(go.Scatter(
    x=df['cuota'], y=df['pct_interes_interes_total'],
    mode="lines+markers", name="% Interés/Total interés",
    line=dict(color="red", dash="dot"),
    hovertext=df['periodo'].dt.strftime("%Y-%m")
))
fig3.add_trace(go.Scatter(
    x=df['cuota'], y=df['pct_interes_acomulado_interes_total'],
    mode="lines+markers", name="% Interés acumulado/Total interés",
    line=dict(color="purple", dash="dot"),
    hovertext=df['periodo'].dt.strftime("%Y-%m")
))

fig3.update_layout(
    title="Evolución de Tasas y Proporciones",
    xaxis_title="Cuota",
    yaxis_title="Proporción (%)",
    yaxis_tickformat=".2%",
    template="plotly_white"
)

# Mostrar
fig1.show()
fig2.show()
fig3.show()


KeyError: 'cuota'

In [39]:
df_sample = df[0:13]
df_sample

,periodo,amortizacion,interes,pago_mensual,saldo
0,2021-11,0.00,0.00,0.00,"90,000,000.00"
1,2021-12,"791,952.54","627,008.06","1,418,960.61","89,208,047.46"
2,2022-01,"797,469.88","621,490.72","1,418,960.61","88,410,577.57"
3,2022-02,"803,025.66","615,934.94","1,418,960.61","87,607,551.91"
4,2022-03,"808,620.15","610,340.46","1,418,960.61","86,798,931.76"
5,2022-04,"814,253.61","604,707.00","1,418,960.61","85,984,678.16"
6,2022-05,"819,926.31","599,034.29","1,418,960.61","85,164,751.85"
7,2022-06,"825,638.54","593,322.07","1,418,960.61","84,339,113.31"
8,2022-07,"831,390.56","587,570.05","1,418,960.61","83,507,722.75"
9,2022-08,"837,182.66","581,777.95","1,418,960.61","82,670,540.09"


In [50]:
90_000_000*0.08688

7819200.0

In [49]:
((627008.06*12)/90_000_000)*100

8.360107466666667

In [45]:
df_sample.pago_mensual.sum()

np.float64(17027527.319999997)

In [44]:
df_sample.amortizacion.sum()

np.float64(9876165.379999999)

In [40]:
df_sample.interes.sum()

np.float64(7151361.88)

In [46]:
df_sample.interes.sum()/df_sample.pago_mensual.sum()

np.float64(0.41998827813362166)

In [43]:
round(df_sample.interes.sum()/90_000_000,6)*100

np.float64(7.946000000000001)

In [9]:
df.amortizacion.sum()

np.float64(90000000.0)

In [7]:
df.interes.sum()

np.float64(29192690.95)

# 02. Prestamo actual

In [17]:
df_primera_parte = df[0:25]
df_primera_parte

,periodo,amortizacion,interes,pago_mensual,saldo
0,2021-11,0.00,0.00,0.00,"90,000,000.00"
1,2021-12,"791,952.54","627,008.06","1,418,960.61","89,208,047.46"
2,2022-01,"797,469.88","621,490.72","1,418,960.61","88,410,577.57"
3,2022-02,"803,025.66","615,934.94","1,418,960.61","87,607,551.91"
4,2022-03,"808,620.15","610,340.46","1,418,960.61","86,798,931.76"
5,2022-04,"814,253.61","604,707.00","1,418,960.61","85,984,678.16"
6,2022-05,"819,926.31","599,034.29","1,418,960.61","85,164,751.85"
7,2022-06,"825,638.54","593,322.07","1,418,960.61","84,339,113.31"
8,2022-07,"831,390.56","587,570.05","1,418,960.61","83,507,722.75"
9,2022-08,"837,182.66","581,777.95","1,418,960.61","82,670,540.09"


In [19]:
df_02 = tabla_mensual_amortizacion_francesa(
    capital_inicial=39_256_073.95,
    tasa=0.08688,
    periodos_n = 31,
    periodos_temporalidad = 'mensual',
    tasa_temporalidad='anual',
    tasa_tipo='efectiva'
)

fecha_inicio = '2025-06-01'

# Crear rango de fechas mensuales
fechas = pd.date_range(start=fecha_inicio, periods=len(df), freq='MS')  # 'MS' = Month Start

# Crear nueva columna con formato 'YYYY-MM'
df_02['periodo'] = fechas.strftime('%Y-%m')
df_02

x = 2 39256073.95


,periodo,amortizacion,interes,pago_mensual,saldo
0,2025-06,0.00,0.00,0.00,"39,256,073.95"
1,2025-07,"1,138,888.43","273,487.50","1,412,375.93","38,117,185.52"
2,2025-08,"1,146,822.79","265,553.14","1,412,375.93","36,970,362.73"
3,2025-09,"1,154,812.42","257,563.51","1,412,375.93","35,815,550.31"
4,2025-10,"1,162,857.72","249,518.21","1,412,375.93","34,652,692.58"
5,2025-11,"1,170,959.07","241,416.86","1,412,375.93","33,481,733.52"
6,2025-12,"1,179,116.85","233,259.08","1,412,375.93","32,302,616.66"
7,2026-01,"1,187,331.47","225,044.46","1,412,375.93","31,115,285.19"
8,2026-02,"1,195,603.32","216,772.61","1,412,375.93","29,919,681.87"
9,2026-03,"1,203,932.80","208,443.13","1,412,375.93","28,715,749.07"


In [20]:
1145154.86+265576.14+6913.00+29674.00

1447318.0

In [24]:
df_2025_09_to_end = df_02[3:]
df_2025_09_to_end

,periodo,amortizacion,interes,pago_mensual,saldo
3,2025-09,"1,154,812.42","257,563.51","1,412,375.93","35,815,550.31"
4,2025-10,"1,162,857.72","249,518.21","1,412,375.93","34,652,692.58"
5,2025-11,"1,170,959.07","241,416.86","1,412,375.93","33,481,733.52"
6,2025-12,"1,179,116.85","233,259.08","1,412,375.93","32,302,616.66"
7,2026-01,"1,187,331.47","225,044.46","1,412,375.93","31,115,285.19"
8,2026-02,"1,195,603.32","216,772.61","1,412,375.93","29,919,681.87"
9,2026-03,"1,203,932.80","208,443.13","1,412,375.93","28,715,749.07"
10,2026-04,"1,212,320.31","200,055.62","1,412,375.93","27,503,428.76"
11,2026-05,"1,220,766.25","191,609.68","1,412,375.93","26,282,662.52"
12,2026-06,"1,229,271.03","183,104.90","1,412,375.93","25,053,391.49"


In [25]:
df_2025_09_to_end.amortizacion.sum()

np.float64(36970362.73)

In [26]:
df_2025_09_to_end.interes.sum()

np.float64(3988539.2399999998)

In [27]:
df_2025_09_to_end.pago_mensual.sum()

np.float64(40958901.97)

In [32]:
df_2025_09_to_end.interes.sum()/df_2025_09_to_end.pago_mensual.sum()

np.float64(0.09737905676576417)

In [31]:
df_2025_09_to_end.interes.sum()/df_2025_09_to_end.amortizacion.sum()

np.float64(0.10788477432934292)

In [30]:
df_2025_09_to_end.amortizacion.sum()/df_2025_09_to_end.pago_mensual.sum()

np.float64(0.9026209432342358)